In [ ]:
import subprocess, sys, os

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch==2.2.0+cu118", "torchvision==0.17.0+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118", "-q"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "numpy==1.26.4", "--force-reinstall", "-q"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "ultralytics==8.3.0", "-q"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "ray", "-y"
])

os.execv(sys.executable, [sys.executable] + sys.argv)

In [1]:
import yaml, glob, time, json, os
import torch
from ultralytics import YOLO

os.environ["WANDB_DISABLED"] = "true"
os.environ["TUNE_DISABLE_AUTO_CALLBACK_LOGGERS"] = "1"

DATASET_PATH = "/kaggle/input/datasets/youssefahmed003/junk-food-object-detection-dataset-yolo-format/yolo_dataset"

label_files = glob.glob(f"{DATASET_PATH}/labels/train/*.txt")
class_ids = set()
for f in label_files[:500]:
    with open(f) as fp:
        for line in fp:
            class_ids.add(int(line.split()[0]))
nc = max(class_ids) + 1

names = ["burger","candy","chips","chocolate","donut","energy_drink","french_fries","fried_chicken","hot_dog","ice_cream","nachos","onion_rings","pizza","popcorn","soda","taco"]
if len(names) != nc:
    names = [str(i) for i in range(nc)]

yaml_path = "/kaggle/working/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump({"path": DATASET_PATH, "train": "images/train", "val": "images/val", "test": "images/test", "nc": nc, "names": names}, f, default_flow_style=False)
print(f"✅ data.yaml 生成完成，Classes: {nc}")

model = YOLO("yolov8s.pt")
start = time.time()
model.train(
    project="runs", data=yaml_path, epochs=50,
    imgsz=640, batch=16, device=0, name="aug_basic", exist_ok=True, verbose=False,
    fliplr=0.5, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, mosaic=0.0, mixup=0.0,
)
elapsed = time.time() - start
print(f"✅ 训练完成，用时 {elapsed/60:.1f} 分钟")

weight_path = "/kaggle/working/runs/aug_basic/weights/best.pt"
if os.path.exists(weight_path):
    print(f"✅ 找到模型: {weight_path} ({os.path.getsize(weight_path)/1024/1024:.1f} MB)")
else:
    print("❌ 找不到权重文件")
    for root, dirs, files in os.walk('/kaggle/working/'):
        for f in files:
            if f.endswith('.pt'):
                print(f"找到: {os.path.join(root, f)}")

✅ data.yaml 生成完成，Classes: 35
New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.2.0+cu118 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/kaggle/working/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=runs, name=aug_basic, exist_ok=True, pretrained=True, optimizer=auto, verbose=False, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=Fal

E0000 00:00:1776476795.353228     609 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776476795.360426     609 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776476795.377879     609 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776476795.377898     609 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776476795.377901     609 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776476795.377903     609 computation_placer.cc:177] computation placer already registered. Please check linka

Overriding model.yaml nc=80 with nc=35

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytic

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


...


50 epochs completed in 2.286 hours.
Optimizer stripped from runs/aug_basic/weights/last.pt, 20.0MB
Optimizer stripped from runs/aug_basic/weights/best.pt, 20.0MB

Validating runs/aug_basic/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.2.0+cu118 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 186 layers, 9,841,209 parameters, 0 gradients, 23.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 33/33 [00:08<00:00,  3.73it/s]


                   all       1039       1479      0.876      0.896      0.921      0.699
Speed: 0.2ms preprocess, 4.4ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/aug_basic


lr/pg0,▃▆███▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg1,▃▆███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
lr/pg2,▃▆███▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
metrics/mAP50(B),▁▄▄▅▆▇▇▇▇▇▇█▇▇▇█████████████████████████
metrics/mAP50-95(B),▁▃▃▄▅▆▆▆▆▆▇▇▇▇▇▇█▇█▇████████████████████
metrics/precision(B),▁▅▅▅▅▆▇▇▆▆▇▇▇▇▇▇▇█▇██▇▇█████████████████
metrics/recall(B),▁▃▄▄▅▆▆▇▇▇▇▇▇█▇███▇█████████████████████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


✅ 训练完成，用时 138.4 分钟
✅ 找到模型: /kaggle/working/runs/aug_basic/weights/best.pt (19.0 MB)
